# FX Fintech Product Analytics Workbook

Synthetic product behavior is combined with historical Yahoo Finance FX observations. Product results are illustrative, not production evidence.

## 0.1 Import Libraries

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics_workflow import (
    CSV_DIR, build_data_mart, dataset_overview, export_product_analysis,
    project_setup, run_campaign_simulator, run_data_quality_checks,
    run_fx_readiness_model, run_simulated_ab_test, train_repeat_models,
)
pd.set_option("display.max_columns", 30)

## 0.2 Project Configuration

In [2]:
project_setup()
build_data_mart()
print(f"Project root: {PROJECT_ROOT}")
print("Random seed: 42")
print("Product source: synthetic SQLite database")
print("FX model source: cached Yahoo Finance historical data")

Project root: C:\Users\Anwar\Documents\Work duty\switchwon
Random seed: 42
Product source: synthetic SQLite database
FX model source: cached Yahoo Finance historical data


## 1.1 Dataset Overview

The SQLite product tables are synthetic. The legacy hourly FX table is simulated and is not used to train the volatility model; that model uses a cached Yahoo Finance snapshot.

In [3]:
overview = dataset_overview()
overview

,table,rows
0,users,45000
1,events,2204400
2,transactions,261679
3,support_tickets,18000
4,fx_rates_hourly,70176
5,marketing_spend,9238
6,modeling_user_month_snapshots,206376


## 1.2 Data Quality Checks

Checks distinguish invalid records from expected optional fields and review-only outliers.

In [4]:
quality_summary, quality_issues = run_data_quality_checks()
quality_summary

,severity,checks,checks_with_issues,affected_rows
0,all,20,11,4219556
1,error,10,1,1675
2,review,1,1,7337
3,warning,9,9,4210544


In [5]:
quality_issues[quality_issues['issue_count'] > 0]

,check,table,column,issue_count,severity,detail
0,missing_values,users,campaign_id,27115,warning,Null or blank values; review whether the field...
1,missing_values,users,kyc_completed_at,11594,warning,Null or blank values; review whether the field...
2,missing_values,users,bank_account_linked_at,15710,warning,Null or blank values; review whether the field...
3,missing_values,events,device_os,875406,warning,Null or blank values; review whether the field...
4,missing_values,events,currency_pair,894108,warning,Null or blank values; review whether the field...
5,missing_values,events,campaign_id,2120532,warning,Null or blank values; review whether the field...
6,missing_values,transactions,executed_rate,20683,warning,Null or blank values; review whether the field...
7,missing_values,transactions,failure_reason,240996,warning,Null or blank values; review whether the field...
15,nonpositive_transaction_volume,transactions,transaction_amount_krw,1675,error,Transaction amount should be positive.
17,duplicate_event_rows,events,"user_id, session_id, event_timestamp, event_name",4400,warning,Extra duplicated event-key rows; clean_events ...


## 1.3 Product Funnel Analysis

Activation requires KYC, bank link, and a first successful exchange within 14 days.

In [6]:
product_outputs = export_product_analysis()
funnel = product_outputs['product_funnel.csv']
funnel

,valid_signups,kyc_completed,bank_linked,first_successful_exchange,activated_within_14_days
0,44974,33406,29264,25791,9649


## 1.4 Cohort Retention Analysis

Monthly retention starts from each user's first successful exchange month.

In [7]:
cohort = product_outputs["cohort_retention.csv"]
cohort_pivot = cohort[cohort["months_since_first_transaction"].between(1, 6)].pivot(
    index="cohort_month", columns="months_since_first_transaction", values="retention_rate"
)
cohort_pivot.tail(12)

months_since_first_transaction,1,2,3,4,5,6
cohort_month,,,,,,
2024-12-01,44.47,39.88,42.77,42.26,42.86,42.35
2025-01-01,42.41,40.11,40.11,43.05,40.57,43.88
2025-02-01,43.90,44.43,44.22,41.76,41.86,43.36
2025-03-01,44.94,46.58,43.29,46.06,45.45,43.03
2025-04-01,49.09,48.04,50.49,51.33,46.78,45.66
2025-05-01,49.62,49.23,47.24,48.54,43.93,44.93
2025-06-01,51.10,49.33,50.25,49.12,46.43,42.46
2025-07-01,58.66,56.66,57.19,53.08,49.74,NaN
2025-08-01,55.52,56.09,51.97,51.46,NaN,NaN


## 1.5 Feature Adoption Analysis

Adoption is descriptive. It does not establish that feature use causes retention.

In [8]:
feature_adoption = product_outputs['feature_adoption.csv']
feature_adoption

,feature,adopted_users,eligible_users,adoption_rate
0,rate_alert,13808,44974,30.70
1,target_rate_order,15595,44974,34.68


In [9]:
product_outputs['feature_retention_comparison.csv']

,comparison_type,user_group,first_exchange_users,d30_repeat_users,d30_repeat_rate
0,Rate-alert adoption,Rate-alert users,13777,7555,54.84
1,Rate-alert adoption,Non-rate-alert users,12014,3208,26.70
2,Target-rate adoption,Target-rate users,15595,8110,52.00
3,Target-rate adoption,Manual-only users,10196,2653,26.02


## 1.6 Acquisition Channel Quality

Channel quality combines activation and D30 repeat behavior rather than signup volume alone.

In [10]:
channel_quality = product_outputs["funnel_by_acquisition_channel.csv"].merge(
    product_outputs["d30_repeat_by_channel.csv"], on="acquisition_channel", how="left"
)
channel_quality.sort_values(["activation_rate_14d", "d30_repeat_rate"], ascending=False)

,acquisition_channel,valid_signups,kyc_completed,kyc_completion_rate,bank_linked,bank_link_rate,first_successful_exchange,first_exchange_rate,activated_within_14_days,activation_rate_14d,first_exchange_users,d30_repeat_users,d30_repeat_rate
0,referral,8154,6573,80.61,5769,70.75,5076,62.25,1948,23.89,5076,2114,41.65
1,direct,3984,2977,74.72,2600,65.26,2307,57.91,898,22.54,2307,977,42.35
2,affiliate,4478,3372,75.30,2987,66.70,2611,58.31,994,22.20,2611,1091,41.78
3,content,2676,1986,74.22,1737,64.91,1521,56.84,571,21.34,1521,647,42.54
4,organic_search,13581,10097,74.35,8795,64.76,7761,57.15,2839,20.90,7761,3208,41.33
5,other,1387,1021,73.61,890,64.17,786,56.67,281,20.26,786,309,39.31
6,paid_social,10714,7380,68.88,6486,60.54,5729,53.47,2118,19.77,5729,2417,42.19


## 2.1 User Repeat Prediction

Features summarize the trailing 90 days through each observation date. The target is repeat behavior in the following 30 days.

In [11]:
modeling_check = product_outputs["modeling_table_check.csv"]
modeling_check

,snapshot_rows,users,min_observation_date,max_observation_date,positive_targets,positive_target_rate,avg_recency_days,avg_transactions_90d,avg_failed_ratio_90d
0,206376,23387,2024-06-30,2025-11-30,84459,40.92,63.51,2.28,0.0526


## 2.2 Model Training and Comparison

The split is chronological. Optional libraries are included only when installed.

In [12]:
repeat_results = train_repeat_models()
repeat_results['metrics']

,model,roc_auc,pr_auc,accuracy,precision,recall,f1,decision_threshold,train_end,test_start,selected_model_flag,selection_rationale
0,Logistic Regression,0.771699,0.747299,0.675198,0.595768,0.783553,0.676878,0.438841,2025-08-30,2025-08-31,1,Selected for explainability and stable ranking...
1,Random Forest,0.763541,0.740302,0.656054,0.574814,0.798351,0.668388,0.402779,2025-08-30,2025-08-31,0,
2,XGBoost,0.773027,0.748458,0.674081,0.594210,0.786319,0.676898,0.425678,2025-08-30,2025-08-31,0,
3,LightGBM,0.772578,0.747740,0.676891,0.597916,0.781036,0.677317,0.435730,2025-08-30,2025-08-31,0,


## 2.3 Model Evaluation

Selection balances ranking, positive-class performance, reliability, and explanation cost.

In [13]:
selected_repeat_model = repeat_results["selected_model"]
print(selected_repeat_model)
print(repeat_results["selection_rationale"])
pd.read_csv(CSV_DIR / "user_repeat_confusion_matrix.csv")

Logistic Regression
Selected for explainability and stable ranking; more complex models did not materially improve validation performance.


,actual,Predicted no repeat,Predicted repeat
0,Actual no repeat,27894,19220
1,Actual repeat,7825,28327


In [14]:
pd.read_csv(CSV_DIR / 'user_repeat_classification_report.csv')

,class,precision,recall,f1-score,support
0,0,0.780929,0.592053,0.673500,47114.000000
1,1,0.595768,0.783553,0.676878,36152.000000
2,accuracy,0.675198,0.675198,0.675198,0.675198
3,macro avg,0.688349,0.687803,0.675189,83266.000000
4,weighted avg,0.700537,0.675198,0.674966,83266.000000


## 2.4 Feature Importance

Importance explains model dependence, not causal effect.

In [15]:
pd.read_csv(CSV_DIR / 'user_repeat_feature_importance.csv')

,feature,importance,method
0,transactions_90d,0.856201,absolute standardized coefficient
1,recency_days,0.223953,absolute standardized coefficient
2,failed_ratio_90d,0.130633,absolute standardized coefficient
3,total_volume_90d,0.059511,absolute standardized coefficient


## 2.5 Prediction-to-Action Mapping

Rules convert scores and operational context into testable CRM actions.

In [16]:
targeting = repeat_results["targeting"]
targeting.groupby(["risk_segment", "recommended_action"], as_index=False).agg(
    users=("user_id", "count"),
    avg_repeat_probability=("repeat_probability", "mean"),
).sort_values("users", ascending=False)

,risk_segment,recommended_action,users,avg_repeat_probability
3,Medium repeat probability users,Maintain engagement,10647,0.498072
4,No recent transaction users,Send reactivation education,5978,0.251697
1,High repeat probability users,Maintain engagement,4401,0.879647
0,High failed transaction users,Resolve transaction friction,2263,0.642111
2,High-value at-risk users,Prioritize service recovery,98,0.442340


## 3.1 FX Volatility Regime Analysis

Historical transaction behavior is compared across the existing market-regime labels.

In [17]:
product_outputs['fx_regime_behavior.csv']

,market_regime_at_transaction,transaction_attempts,successful_transactions,success_rate,avg_transaction_amount_krw,completed_volume_krw
0,low,117445,109205,92.98,2251334.0,2.460041e+11
1,normal,88561,81333,91.84,2266686.0,1.846910e+11
2,high,53998,48783,90.34,2264756.0,1.103559e+11


## 3.2 FX Volatility Prediction

Historical Yahoo Finance observations are cached locally. A time split predicts the next day's volatility regime from lagged FX features; this is not an exchange-rate forecast.

In [18]:
fx_results = run_fx_readiness_model()
fx_results['source']

,provider,library,tickers,first_observation,last_observation,rows,cache_path,load_mode,data_classification
0,Yahoo Finance,yfinance,"USDKRW:USDKRW=X, JPYKRW:JPYKRW=X, EURKRW:EURKR...",2016-01-01,2026-07-09,2737,data/external/yahoo_fx_daily.csv,local_cache,external historical market data; not synthetic


In [19]:
fx_results['metrics']

,regime_definition,model,accuracy,balanced_accuracy,precision_macro,recall_macro,f1_macro,selected_model_flag
3,KMeans volatility clusters,Random Forest,0.558442,0.599654,0.531827,0.599654,0.540969,1
2,KMeans volatility clusters,Logistic Regression,0.556586,0.599163,0.516585,0.599163,0.532117,0
1,Quantile tercile,Random Forest,0.525046,0.516742,0.508122,0.516742,0.504733,0
0,Quantile tercile,Logistic Regression,0.506494,0.502964,0.494305,0.502964,0.497038,0


## 3.3 Operational Readiness Signal

The signal supports staffing, transaction monitoring, rate-change messaging, and campaign timing.

In [20]:
fx_results['predictions'].tail(10)

,date,actual_regime,predicted_regime,prob_high,prob_low,prob_normal,recommended_action
529,2026-06-25,low,normal,0.203782,0.381691,0.414527,Maintain standard staffing and transaction mon...
530,2026-06-26,normal,high,0.532640,0.170128,0.297232,"Prepare support coverage, rate-change messagin..."
531,2026-06-29,normal,low,0.225308,0.399445,0.375246,Normal monitoring; use calmer periods for prod...
532,2026-06-30,normal,normal,0.217774,0.349815,0.432411,Maintain standard staffing and transaction mon...
533,2026-07-01,low,low,0.150696,0.466720,0.382584,Normal monitoring; use calmer periods for prod...
534,2026-07-02,normal,high,0.567553,0.147436,0.285011,"Prepare support coverage, rate-change messagin..."
535,2026-07-03,normal,high,0.630588,0.138271,0.231141,"Prepare support coverage, rate-change messagin..."
536,2026-07-06,low,low,0.220798,0.416466,0.362737,Normal monitoring; use calmer periods for prod...
537,2026-07-07,normal,high,0.468209,0.201722,0.330069,"Prepare support coverage, rate-change messagin..."
538,2026-07-08,normal,high,0.552023,0.175010,0.272966,"Prepare support coverage, rate-change messagin..."


## 4.1 Campaign Targeting Logic

Priority rules identify service recovery, reactivation, transaction-friction, and controlled-test audiences.

In [21]:
targeting.groupby("risk_segment", as_index=False).agg(
    users=("user_id", "count"),
    avg_repeat_probability=("repeat_probability", "mean"),
    recent_volume_krw=("total_volume_90d", "sum"),
).sort_values("users", ascending=False)

,risk_segment,users,avg_repeat_probability,recent_volume_krw
3,Medium repeat probability users,10647,0.498072,35681903361
4,No recent transaction users,5978,0.251697,0
1,High repeat probability users,4401,0.879647,104830815333
0,High failed transaction users,2263,0.642111,13639400464
2,High-value at-risk users,98,0.442340,1425808775


## 4.2 Campaign Simulator

Low, base, and high assumptions show sensitivity. Fee impact is a value proxy, not revenue.

In [22]:
campaign_base, campaign_sensitivity = run_campaign_simulator(targeting)
campaign_base

,audience_users,baseline_expected_repeat_users,uplift_scenario,absolute_uplift_assumption,incremental_repeat_users,median_recent_volume_proxy_krw,incremental_volume_proxy_krw,fee_proxy_scenario,fee_proxy_rate_assumption,value_impact_proxy_krw,assumption_note
4,18986,8304.068237,base,0.05,949.3,2388515.5,2.267418e+09,base,0.00025,566854.441038,Directional scenario only; not forecast revenu...


In [23]:
campaign_sensitivity

,audience_users,baseline_expected_repeat_users,uplift_scenario,absolute_uplift_assumption,incremental_repeat_users,median_recent_volume_proxy_krw,incremental_volume_proxy_krw,fee_proxy_scenario,fee_proxy_rate_assumption,value_impact_proxy_krw,assumption_note
0,18986,8304.068237,low,0.02,379.72,2388515.5,9.069671e+08,low,0.00015,1.360451e+05,Directional scenario only; not forecast revenu...
1,18986,8304.068237,low,0.02,379.72,2388515.5,9.069671e+08,base,0.00025,2.267418e+05,Directional scenario only; not forecast revenu...
2,18986,8304.068237,low,0.02,379.72,2388515.5,9.069671e+08,high,0.00035,3.174385e+05,Directional scenario only; not forecast revenu...
3,18986,8304.068237,base,0.05,949.30,2388515.5,2.267418e+09,low,0.00015,3.401127e+05,Directional scenario only; not forecast revenu...
4,18986,8304.068237,base,0.05,949.30,2388515.5,2.267418e+09,base,0.00025,5.668544e+05,Directional scenario only; not forecast revenu...
5,18986,8304.068237,base,0.05,949.30,2388515.5,2.267418e+09,high,0.00035,7.935962e+05,Directional scenario only; not forecast revenu...
6,18986,8304.068237,high,0.08,1518.88,2388515.5,3.627868e+09,low,0.00015,5.441803e+05,Directional scenario only; not forecast revenu...
7,18986,8304.068237,high,0.08,1518.88,2388515.5,3.627868e+09,base,0.00025,9.069671e+05,Directional scenario only; not forecast revenu...
8,18986,8304.068237,high,0.08,1518.88,2388515.5,3.627868e+09,high,0.00035,1.269754e+06,Directional scenario only; not forecast revenu...


## 5.1 A/B Test Design

No production experiment table exists. Assignment and outcomes are simulated to demonstrate analysis logic, not causal evidence.

In [24]:
ab_design, ab_results, ab_segments = run_simulated_ab_test()
ab_design

,field,definition
0,data_source,Simulated randomized assignment and outcomes; ...
1,unit_of_randomization,User
2,control,Existing onboarding with no feature prompt
3,treatment,Onboarding prompt introducing rate-alert or ta...
4,primary_metric,Feature adoption within 7 days
5,secondary_metric,First exchange within 14 days
6,guardrail_metric,Failed transaction or support ticket within 30...
7,minimum_practical_effect,2 percentage-point absolute uplift in the prim...


## 5.2 A/B Test Statistical Evaluation

Report confidence intervals, p-values, and practical significance together.

In [25]:
ab_results

,metric,metric_role,control_n,treatment_n,control_rate,treatment_rate,absolute_uplift,relative_uplift,ci_95_low,ci_95_high,z_statistic,p_value,statistically_significant_5pct,practically_significant,data_source
0,feature_adoption_7d,primary,22425,22549,0.216366,0.264003,0.047637,0.220170,0.039754,0.055521,11.823011,0.000000,True,True,simulated
1,first_exchange_14d,secondary,22425,22549,0.210479,0.223114,0.012635,0.060028,0.005019,0.020250,3.251164,0.001149,True,True,simulated
2,guardrail_event_30d,guardrail,22425,22549,0.073266,0.070824,-0.002443,-0.033343,-0.007222,0.002337,-1.001843,0.316420,False,False,simulated


In [26]:
ab_segments.sort_values('absolute_uplift', ascending=False)

,segment_type,segment,metric,control_n,treatment_n,control_rate,treatment_rate,absolute_uplift,relative_uplift,ci_95_low,ci_95_high,z_statistic,p_value,statistically_significant_5pct,data_source
1,acquisition_channel,content,feature_adoption_7d,1328,1348,0.205572,0.267062,0.061490,0.299116,0.029392,0.093588,3.742447,1.822369e-04,True,simulated
6,acquisition_channel,referral,feature_adoption_7d,4102,4052,0.230619,0.285785,0.055166,0.239206,0.036200,0.074131,5.692271,1.253604e-08,True,simulated
5,acquisition_channel,paid_social,feature_adoption_7d,5350,5364,0.195327,0.244221,0.048894,0.250317,0.033240,0.064548,6.110500,9.931953e-10,True,simulated
3,acquisition_channel,organic_search,feature_adoption_7d,6759,6822,0.221778,0.270595,0.048817,0.220115,0.034352,0.063282,6.601897,4.059286e-11,True,simulated
7,value_segment,High value,feature_adoption_7d,6071,6093,0.220227,0.268505,0.048278,0.219217,0.033030,0.063526,6.195126,5.823850e-10,True,simulated
8,value_segment,Standard value,feature_adoption_7d,16354,16456,0.214932,0.262336,0.047404,0.220552,0.038194,0.056613,10.071061,0.000000e+00,True,simulated
0,acquisition_channel,affiliate,feature_adoption_7d,2161,2317,0.209625,0.249892,0.040267,0.192090,0.015664,0.064870,3.197301,1.387203e-03,True,simulated
2,acquisition_channel,direct,feature_adoption_7d,2008,1976,0.239044,0.277328,0.038284,0.160155,0.011125,0.065444,2.761244,5.758165e-03,True,simulated
4,acquisition_channel,other,feature_adoption_7d,717,670,0.217573,0.226866,0.009292,0.042710,-0.034500,0.053085,0.416082,6.773497e-01,False,simulated


## 6.1 Dashboard Export Files

All dashboard inputs are written to `outputs/csv/`; trained models are written to `models/`.

In [27]:
exports = pd.DataFrame({
    "file": [path.name for path in sorted(CSV_DIR.glob("*.csv"))],
    "size_kb": [round(path.stat().st_size / 1024, 1) for path in sorted(CSV_DIR.glob("*.csv"))],
})
exports

,file,size_kb
0,ab_test_design.csv,0.5
1,ab_test_results.csv,0.8
2,ab_test_segment_results.csv,2.2
3,campaign_sensitivity_analysis.csv,1.7
4,campaign_simulator_outputs.csv,0.4
5,cohort_retention.csv,8.4
6,customer_value_segmentation.csv,0.3
7,d30_repeat_by_channel.csv,0.3
8,data_quality_issues.csv,2.5
9,data_quality_summary.csv,0.1


## 6.2 Limitations and Next Steps

- Product behavior is synthetic; findings do not represent company performance.
- FX prices are historical Yahoo Finance observations, subject to provider availability and adjustments.
- Feature adoption comparisons are observational and may reflect user intent.
- The experiment is simulated and must not be presented as production causal evidence.
- Fee impact is an assumption-based proxy, not reported revenue.
- Next steps are production metric validation, experiment instrumentation, probability calibration, and drift monitoring.